In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from matplotlib import pyplot as plt
from keras.datasets import fashion_mnist
from keras.models import Sequential

## Завдання 1

In [2]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape([-1, 28, 28, 1])
x_test = x_test.reshape([-1, 28, 28, 1])

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [3]:
model = Sequential([
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation='relu'),

    keras.layers.Dense(10, activation='softmax')
])

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [5]:
model.fit(x_train, y_train, epochs=50, batch_size=64, validation_data=(x_test, y_test), callbacks=[reduce_lr])

Epoch 1/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.8182 - loss: 0.4926 - val_accuracy: 0.8682 - val_loss: 0.3578 - learning_rate: 0.0010
Epoch 2/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8770 - loss: 0.3339 - val_accuracy: 0.8757 - val_loss: 0.3272 - learning_rate: 0.0010
Epoch 3/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8923 - loss: 0.2926 - val_accuracy: 0.8979 - val_loss: 0.2823 - learning_rate: 0.0010
Epoch 4/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9015 - loss: 0.2688 - val_accuracy: 0.8937 - val_loss: 0.2914 - learning_rate: 0.0010
Epoch 5/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9064 - loss: 0.2489 - val_accuracy: 0.8874 - val_loss: 0.2969 - learning_rate: 0.0010
Epoch 6/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9118 - loss: 0.2371 - val_accuracy: 0.9042 - val_loss: 0.2606 - learning_rate: 0.0010
Epoch 7/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9180 - loss: 0.2222 

In [7]:
model.save('vgg16_model.keras')
from google.colab import files
files.download('vgg16_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Висновок:

У ході виконання завдання була побудована згорткова нейронна мережа для класифікації товарів з датасету fashion_mnist. В процесі пошуку оптимальної архітектури були перевірені різні комбінації гіперпараметрів: кількість фільтрів на шарах Conv2D (на останньому шарі випробував 64 та 128) та кількість нейронів у Dense шарі (64, 128 та 256). Серед оптимізаторів були перевірені RMSprop та Adam. Adam показав кращий результат у комбінації з callback ReduceLROnPlateau, який автоматично зменшував learning rate коли val_loss переставав покращуватись — без нього val_loss та val_accuracy нестабільно коливались. Також була використана регуляризація у вигляді BatchNormalization та Dropout(0.5), що допомогло зменшити overfitting.
Фінальна архітектура: три блоки Conv2D (32→64→128 фільтрів) з BatchNormalization та MaxPooling після кожного, Dropout(0.5) після Flatten, Dense(128) та вихідний шар Dense(10, softmax). Оптимізатор — Adam з ReduceLROnPlateau.
Повнозв'язна мережа з попереднього ДЗ показала максимальний результат 90.73%, тоді як CNN досягла 93.01% на епосі 26 — приріст 2.28%. Це підтверджує що згорткові мережі краще підходять для задач класифікації зображень завдяки здатності вловлювати просторові патерни — форму, краї та текстури об'єктів.


## Завдання 2

In [8]:
from keras.applications.vgg16 import preprocess_input, VGG16

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

x_train = tf.image.grayscale_to_rgb(tf.expand_dims(x_train, -1))
x_test = tf.image.grayscale_to_rgb(tf.expand_dims(x_test, -1))

x_train = tf.image.resize(x_train, [72, 72])
x_test = tf.image.resize(x_test, [72, 72])

x_train = preprocess_input(x_train)
x_test = preprocess_input(x_test)

In [9]:
conv_base = VGG16(weights="imagenet", include_top=False, input_shape=(72, 72, 3))
conv_base.trainable = False

model = Sequential([
    conv_base,
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')
])

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [10]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(x_test, y_test)
)

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 70s 69ms/step - accuracy: 0.7951 - loss: 0.8166 - val_accuracy: 0.8602 - val_loss: 0.3824
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 59s 63ms/step - accuracy: 0.8458 - loss: 0.4390 - val_accuracy: 0.8740 - val_loss: 0.3538
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 61s 65ms/step - accuracy: 0.8591 - loss: 0.3964 - val_accuracy: 0.8782 - val_loss: 0.3436
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 62s 66ms/step - accuracy: 0.8651 - loss: 0.3823 - val_accuracy: 0.8826 - val_loss: 0.3377
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 62s 66ms/step - accuracy: 0.8710 - loss: 0.3564 - val_accuracy: 0.8830 - val_loss: 0.3416
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 62s 66ms/step - accuracy: 0.8758 - loss: 0.3443 - val_accuracy: 0.8811 - val_loss: 0.3407
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 62s 66ms/step - accuracy: 0.8795 - loss: 0.3340 - val_accuracy: 0.8832 - val_loss: 0.3427
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 62s 66ms/step - accuracy: 0.8824 - loss: 0.3305 - 

In [12]:
conv_base.trainable = True
for layer in conv_base.layers:
    if "block5" in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False

In [13]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5), # 0.00001
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [14]:
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
)

In [15]:
history_fine = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=[reduce_lr]
)

Epoch 1/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 90s 90ms/step - accuracy: 0.9007 - loss: 0.2675 - val_accuracy: 0.9031 - val_loss: 0.2974 - learning_rate: 1.0000e-05
Epoch 2/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 80s 85ms/step - accuracy: 0.9176 - loss: 0.2178 - val_accuracy: 0.9098 - val_loss: 0.2898 - learning_rate: 1.0000e-05
Epoch 3/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 79s 85ms/step - accuracy: 0.9297 - loss: 0.1843 - val_accuracy: 0.9134 - val_loss: 0.2855 - learning_rate: 1.0000e-05
Epoch 4/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 80s 85ms/step - accuracy: 0.9362 - loss: 0.1629 - val_accuracy: 0.9126 - val_loss: 0.3062 - learning_rate: 1.0000e-05
Epoch 5/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 79s 85ms/step - accuracy: 0.9434 - loss: 0.1460 - val_accuracy: 0.9199 - val_loss: 0.2895 - learning_rate: 1.0000e-05
Epoch 6/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 80s 85ms/step - accuracy: 0.9478 - loss: 0.1329 - val_accuracy: 0.9189 - val_loss: 0.2984 - learning_rate: 1.0000e-05
Epoch 7/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 79s 85ms/ste

In [16]:
model.save('vgg_model.keras')

from google.colab import files
files.download('vgg_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Висновок:

Використання VGG16 для такої простої задачі, як класифікація Fashion MNIST, наочно демонструє, що потужніша архітектура не завжди гарантує кращий результат, адже моя кастомна CNN (93%) виявилася ефективнішою за важку VGG16 (92.6%). Головна проблема полягала в тому, що VGG16 розроблялася під великі кольорові фотографії, і для її роботи нам довелося штучно розтягувати маленькі зображення 28х28 до 72х72, що неминуче призвело до появи розмитості та втрати дрібних деталей. Крім того, на етапі виділення ознак (Feature Extraction) модель показала посередні результати, і лише завдяки тонкому донавчанню (Fine-tuning) останнього згорткового блоку з мікроскопічним кроком навчання нам вдалося наблизитися до показників простої мережі. У підсумку, хоча Transfer Learning і є незамінним інструментом для складних фотореалістичних об'єктів, для схематичних чорно-білих даних він виявився занадто громіздким, повільним у навчанні та ресурсомістким, підтверджуючи правило, що архітектура має відповідати складності конкретного завдання.